In [2]:
%pip install polars

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.0/828.0 kB 41.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 MB 161.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [polars]2m1/2 [polars]
Note: you may need to restart the kernel to use updated packages.


In [4]:
import polars as pl
import numpy as np
import pandas as pd
from datetime import date
from sklearn.preprocessing import StandardScaler
import gc
import os

In [7]:
file_path = 'data_ready/df_2025_valid_only.parquet'
lazy_df = pl.scan_parquet(file_path)
drop_columns = ['serial_number', 'model', 'datacenter']
lazy_df = lazy_df.drop(drop_columns)

print('Splitting chronologically...')

train_set = lazy_df.filter(pl.col('date') < date(2025, 10, 1))
test_set = lazy_df.filter(pl.col('date') >= date(2025, 10, 1))

print('Downsampling healthy drives in training set...')
train_fail = train_set.filter(pl.col('failure') == 1)
train_healthy = train_set.filter(pl.col('failure') == 0).gather_every(20)

test_fail = test_set.filter(pl.col('failure') == 1)
test_healthy = test_set.filter(pl.col('failure') == 0).gather_every(20)

lazy_train = pl.concat([train_fail, train_healthy])
lazy_test = pl.concat([test_fail, test_healthy])

lazy_train.sink_parquet("data_ready/train_split.parquet")

lazy_test.sink_parquet("data_ready/test_split.parquet")

print("Loading optimized datasets into memory...")
df_train = pl.read_parquet("data_ready/train_split.parquet")
df_test = pl.read_parquet("data_ready/test_split.parquet")

print("Applying final memory-safe downsample...")
train_fail = df_train.filter(pl.col("failure") == 1)
train_healthy = df_train.filter(pl.col("failure") == 0).sample(fraction=0.10, seed=6740)
df_train = pl.concat([train_fail, train_healthy])

test_fail = df_test.filter(pl.col("failure") == 1)
test_healthy = df_test.filter(pl.col("failure") == 0).sample(fraction=0.10, seed=6740)
df_test = pl.concat([test_fail, test_healthy])

print(f"NEW Train Shape: {df_train.shape} | NEW Test Shape: {df_test.shape}")

print("Handling Nulls...")
df_train = df_train.fill_null(0)
df_test = df_test.fill_null(0)

drop_cols = ['date', 'failure'] + [col for col in df_train.columns if 'normalized' in col.lower()]

print('--- Phase 1 Targets (Classification) ---')
y_train = df_train['failure'].to_pandas()
y_test = df_test['failure'].to_pandas()

print('--- Phase 2 Targets (Survival Analysis)')
y_event_train = y_train.copy()
y_time_train = df_train['smart_9_raw'].to_pandas() / 24

y_event_test = y_test.copy()
y_time_test = df_test['smart_9_raw'].to_pandas() / 24

print('--- Preparing Base X Matrices ---')
X_train_pl = df_train.drop(drop_cols)
X_test_pl = df_test.drop(drop_cols)

del df_train
del df_test
gc.collect()

print("Converting X matrices to Pandas...")
X_train = X_train_pl.to_pandas()
X_test = X_test_pl.to_pandas()

del X_train_pl
del X_test_pl
gc.collect()

print('--- Phase 3 Matrices (Anomaly Deteection) ---')
X_train_healthy = X_train[y_train == 0].copy()

scaler = StandardScaler()
X_train_healthy_scaled = pd.DataFrame(scaler.fit_transform(X_train_healthy), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

print("Success! Matrices are ready for Scikit-Learn.")

Splitting chronologically...
Downsampling healthy drives in training set...
Loading optimized datasets into memory...
Applying final memory-safe downsample...
NEW Train Shape: (765101, 174) | NEW Test Shape: (155648, 174)
Handling Nulls...
--- Phase 1 Targets (Classification) ---
--- Phase 2 Targets (Survival Analysis)
--- Preparing Base X Matrices ---
Converting X matrices to Pandas...
--- Phase 3 Matrices (Anomaly Deteection) ---
Success! Matrices are ready for Scikit-Learn.


In [8]:
os.makedirs("data_ready/ml_matrices", exist_ok=True)

print("Saving Phase 1 (Classification) Matrices...")
X_train.to_parquet("data_ready/ml_matrices/X_train.parquet")
X_test.to_parquet("data_ready/ml_matrices/X_test.parquet")
y_train.to_frame().to_parquet("data_ready/ml_matrices/y_train.parquet")
y_test.to_frame().to_parquet("data_ready/ml_matrices/y_test.parquet")

print("Saving Phase 2 (Survival Analysis) Targets...")
y_event_train.to_frame().to_parquet("data_ready/ml_matrices/y_event_train.parquet")
y_time_train.to_frame().to_parquet("data_ready/ml_matrices/y_time_train.parquet")
y_event_test.to_frame().to_parquet("data_ready/ml_matrices/y_event_test.parquet")
y_time_test.to_frame().to_parquet("data_ready/ml_matrices/y_time_test.parquet")

print("Saving Phase 3 (Anomaly Detection) Matrices...")
X_train_healthy_scaled.to_parquet("data_ready/ml_matrices/X_train_healthy_scaled.parquet")
X_test_scaled.to_parquet("data_ready/ml_matrices/X_test_scaled.parquet")

print("Success! All data is securely locked in on your SageMaker disk.")

Saving Phase 1 (Classification) Matrices...
Saving Phase 2 (Survival Analysis) Targets...
Saving Phase 3 (Anomaly Detection) Matrices...
Success! All data is securely locked in on your SageMaker disk.


In [15]:
X_train.head(5)

,capacity_bytes,smart_1_raw,smart_5_raw,smart_9_raw,smart_194_raw,smart_197_raw,smart_2_raw,smart_3_raw,smart_4_raw,smart_7_raw,...,is_legacy_format,smart_71_raw,smart_90_raw,datacenter,cluster_id,pod_slot_num,smart_82_raw,smart_27_raw,smart_211_raw,smart_212_raw
0,8001563222016,111314472.0,22513.0,72366.0,38.0,0.0,0.0,0.0,21.0,2.108343e+09,...,0,0.0,0.0,sac0,0,0,0.0,0.0,0.0,0.0
1,8001563222016,136209792.0,11176.0,65025.0,42.0,0.0,0.0,0.0,40.0,9.516666e+08,...,0,0.0,0.0,phx1,0,0,0.0,0.0,0.0,0.0
2,12000138625024,40867320.0,0.0,33885.0,30.0,0.0,0.0,0.0,12.0,6.462630e+08,...,0,0.0,0.0,sac0,0,50,0.0,0.0,0.0,0.0
3,12000138625024,0.0,10.0,49440.0,29.0,100.0,0.0,419.0,20.0,0.000000e+00,...,0,0.0,0.0,sac2,0,4,0.0,0.0,0.0,0.0
4,16000900661248,0.0,0.0,19241.0,39.0,0.0,100.0,327.0,12.0,0.000000e+00,...,0,0.0,0.0,phx1,0,33,0.0,0.0,0.0,0.0
